In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import DeltaTable

dbutils.widgets.text('incremental_flag','0')
incremental_flag = dbutils.widgets.get('incremental_flag')

In [0]:
df = spark.sql("select * from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`")
df.display()

In [0]:
df_src = spark.sql("select distinct Date_ID,day,month,year from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`")

if spark.catalog.tableExists("car_sales.gold.dim_date"):
    df_sink = spark.sql("select Date_ID,day,month,year,dim_date_key from car_sales.gold.dim_date")
else:
    df_sink = spark.sql("select Date_ID,day,month,year, 0 dim_date_key from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver` where 1=0")
df_sink.display()


In [0]:
df_filter  = df_src.join(df_sink,df_src.Date_ID == df_sink.Date_ID,'left').select(df_src.Date_ID,df_src.day,df_src.month,df_src.year,df_sink.dim_date_key)

df_filter_old = df_filter.where(df_filter['dim_date_key'].isNotNull())
df_filter_new = df_filter.where(df_filter['dim_date_key'].isNull()).select(df_filter.Date_ID,df_filter.day,df_filter.month,df_filter.year)
df_filter_old.display()
df_filter_new.display()


In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    if spark.catalog.tableExists("car_sales.gold.dim_date"):
        df = spark.sql("select max(dim_date_key) from car_sales.gold.dim_date")
        max_value = df.collect()[0][0]
print(max_value)
            

In [0]:
df_filter_new = df_filter_new.withColumn('dim_date_key',max_value + F.monotonically_increasing_id())

df_final = df_filter_new.union(df_filter_old)
df_final.display()